# German Credit Risk: Model, Explainability & Fairness (v2 — 4 โมเดล)

ชุดข้อมูล: Statlog German Credit (OpenML `data_id=31`)

| ส่วน | เนื้อหา |
|---|---|
| 0 | ติดตั้งและนำเข้าไลบรารี |
| 1 | โหลดข้อมูลและสำรวจข้อมูล (EDA) |
| 2 | target, แบ่งข้อมูล, Pipeline **4 อัลกอริทึม**, ปรับ hyperparameter, เลือก threshold |
| 3 | Permutation importance บนชุดทดสอบ |
| 4 | Fairness metrics ตาม sex และ age_group |
| 5 | บันทึก `german_credit.csv`, `model_artifacts.joblib`, `requirements.txt` แล้วดาวน์โหลด |

**วิธีใช้:** Runtime → Run all → ได้ไฟล์ `deploy_files.zip` นำไฟล์ข้างในไปวางรวมกับไฟล์ใน GitHub repository (app.py, train_model.py ฯลฯ)

**สิ่งที่เปลี่ยนจาก v1:** เพิ่มโมเดลที่ 4 (Support Vector Machine), เก็บ OOF probability และ `y_train` เพื่อให้แอปคำนวณ threshold ตามต้นทุนใหม่ได้, แยก `app.py` ออกเป็นไฟล์ใน repository แทนการเขียนด้วย `%%writefile`

## 0. ติดตั้งและนำเข้าไลบรารี

In [ ]:
import warnings, os, shutil
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn

from sklearn.datasets import fetch_openml
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GridSearchCV, cross_val_predict)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (roc_auc_score, confusion_matrix, recall_score,
                             precision_score, f1_score, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance
from sklearn.base import clone

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS = 5
COST_FN, COST_FP = 5, 1   # FN = ปล่อยกู้ให้คนที่ bad, FP = ปฏิเสธคนที่ good
POS_LABEL = "bad"

pd.set_option("display.max_columns", 50)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## 1. โหลดข้อมูลและสำรวจข้อมูล (EDA)

In [ ]:
df = fetch_openml(data_id=31, as_frame=True).frame
for c in df.select_dtypes(include="category").columns:
    df[c] = df[c].astype(str)
print("ขนาดข้อมูล:", df.shape)
df.head()

### 1.1 ชนิดข้อมูลและค่าที่หายไป

In [ ]:
pd.DataFrame({"dtype": df.dtypes.astype(str), "n_unique": df.nunique(), "missing": df.isna().sum()})

### 1.2 สถิติเชิงพรรณนาของตัวแปรตัวเลข

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols + ["class"]]
print("ตัวแปรตัวเลข:", num_cols)
print("ตัวแปรหมวดหมู่:", cat_cols)
df[num_cols].describe().T

### 1.3 การกระจายของ target
ข้อมูลไม่สมดุล (good มากกว่า bad) จึงแบ่งข้อมูลแบบ stratified และใช้ ROC-AUC แทน accuracy

In [ ]:
counts = df["class"].value_counts()
print(counts, "\nสัดส่วน bad =", round((df["class"] == "bad").mean(), 3))
counts.plot(kind="bar", color=["#4C78A8", "#E45756"], title="Class distribution")
plt.ylabel("count"); plt.show()

### 1.4 ตัวแปรตัวเลขแยกตาม class

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), num_cols):
    df.boxplot(column=c, by="class", ax=ax)
    ax.set_title(c); ax.set_xlabel("")
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
plt.suptitle("Numeric features by class"); plt.tight_layout(); plt.show()

### 1.5 อัตรา bad ในแต่ละหมวดของตัวแปรหมวดหมู่ที่สำคัญ

In [ ]:
show = ["checking_status", "credit_history", "savings_status", "purpose"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, c in zip(axes, show):
    (df.groupby(c)["class"].apply(lambda s: (s == "bad").mean())
       .sort_values().plot(kind="barh", ax=ax, title=f"bad rate by {c}"))
plt.tight_layout(); plt.show()

### 1.6 Correlation ของตัวแปรตัวเลข

In [ ]:
corr = df[num_cols].corr()
plt.figure(figsize=(7, 6))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(len(num_cols)), num_cols, rotation=90); plt.yticks(range(len(num_cols)), num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Correlation"); plt.tight_layout(); plt.show()

## 2. โมเดล การอธิบาย และความเป็นธรรม
### 2.1 กำหนด target และแบ่ง train/test แบบ stratified
- `y = 1` เมื่อ `class == 'bad'`
- สร้าง `sex` (จาก personal_status) และ `age_group` (<25 / >=25) สำหรับวัดความเป็นธรรม โดยไม่ใส่ในฟีเจอร์

In [ ]:
df["sex"] = np.where(df["personal_status"].str.startswith("female"), "female", "male")
df["age_group"] = np.where(df["age"] < 25, "<25", ">=25")

y = (df["class"] == POS_LABEL).astype(int)
X = df.drop(columns=["class", "sex", "age_group"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
groups_test = df.loc[X_test.index, ["sex", "age_group"]]

print("Train:", X_train.shape, "bad rate =", round(y_train.mean(), 3))
print("Test :", X_test.shape, "bad rate =", round(y_test.mean(), 3))
print(df[["sex", "age_group"]].value_counts())

### 2.2 Pipeline (ColumnTransformer + โมเดล) 4 อัลกอริทึม
- ตัวเลข → `StandardScaler`, หมวดหมู่ → `OneHotEncoder(handle_unknown="ignore")`
- รวมการแปลงข้อมูลกับโมเดลใน Pipeline เดียว เพื่อป้องกัน data leakage ในการทำ CV
- **โมเดลที่ 4: Support Vector Machine (RBF kernel)** ใช้ `probability=True` เพื่อให้ได้ความน่าจะเป็นสำหรับ ROC-AUC และการเลือก threshold และต้องใช้ StandardScaler (มีอยู่แล้วใน Pipeline)

In [ ]:
def build_preprocessor(X):
    num = X.select_dtypes(include="number").columns.tolist()
    cat = X.select_dtypes(exclude="number").columns.tolist()
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])

model_specs = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, class_weight="balanced"),
        {"model__C": [0.01, 0.1, 1.0, 10.0]}),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1),
        {"model__n_estimators": [200, 400], "model__max_depth": [None, 8],
         "model__min_samples_leaf": [1, 5]}),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {"model__n_estimators": [100, 200], "model__learning_rate": [0.05, 0.1],
         "model__max_depth": [2, 3]}),
    "Support Vector Machine": (
        SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE),
        {"model__C": [0.1, 1.0, 10.0], "model__gamma": ["scale", 0.01]}),
}
print("จำนวนโมเดล:", len(model_specs))

### 2.3 ปรับ hyperparameter ด้วย StratifiedKFold CV (k=5) เฉพาะบนชุดฝึก

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
results = {}
for name, (est, grid) in model_specs.items():
    pipe = Pipeline([("prep", build_preprocessor(X_train)), ("model", est)])
    gs = GridSearchCV(pipe, grid, cv=skf, scoring="roc_auc", n_jobs=-1)
    gs.fit(X_train, y_train)
    results[name] = {
        "pipeline": gs.best_estimator_,
        "best_params": gs.best_params_,
        "cv_auc": gs.best_score_,
        "cv_table": pd.DataFrame(gs.cv_results_)[
            ["params", "mean_test_score", "std_test_score", "rank_test_score"]
        ].sort_values("rank_test_score").reset_index(drop=True),
    }
    print(f"{name:24s} CV AUC = {gs.best_score_:.4f}  best = {gs.best_params_}")

### 2.4 เลือก threshold ที่ลด expected cost (FN:FP = 5:1) จาก out-of-fold predictions
เก็บ `oof_proba` ไว้ด้วย เพื่อให้แอปคำนวณ threshold ใหม่ได้เมื่อผู้ใช้เปลี่ยนสัดส่วนต้นทุน

In [ ]:
def find_best_threshold(y_true, proba, cost_fn=COST_FN, cost_fp=COST_FP):
    rows = []
    for t in np.round(np.arange(0.01, 1.00, 0.01), 2):
        pred = (proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        rows.append({"threshold": t, "FN": int(fn), "FP": int(fp),
                     "cost": int(cost_fn * fn + cost_fp * fp)})
    curve = pd.DataFrame(rows)
    return float(curve.loc[curve["cost"].idxmin(), "threshold"]), curve

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, r in results.items():
    oof = cross_val_predict(r["pipeline"], X_train, y_train, cv=skf, method="predict_proba")[:, 1]
    t, curve = find_best_threshold(y_train.values, oof)
    r.update(oof_proba=oof, oof_auc=roc_auc_score(y_train, oof), threshold=t, cost_curve=curve)
    ax.plot(curve["threshold"], curve["cost"], label=f"{name} (t={t:.2f})")
    print(f"{name:24s} OOF AUC = {r['oof_auc']:.4f}  best threshold = {t:.2f}  "
          f"min OOF cost = {curve['cost'].min()}")
ax.set_xlabel("threshold"); ax.set_ylabel("expected cost (5·FN + FP)")
ax.set_title("Expected cost on out-of-fold predictions"); ax.legend(); plt.show()

### 2.5 ประเมินผลบนชุดทดสอบ
โมเดลที่ใช้ในส่วนถัดไปคือโมเดลที่ CV AUC สูงสุด (เลือกจากชุดฝึก ไม่ใช่จากชุดทดสอบ)

In [ ]:
def cost_of(y_true, pred):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return int(COST_FN * fn + COST_FP * fp)

for name, r in results.items():
    proba = r["pipeline"].predict_proba(X_test)[:, 1]
    pred = (proba >= r["threshold"]).astype(int)
    r.update(proba_test=proba, pred_test=pred, test_metrics={
        "AUC": roc_auc_score(y_test, proba),
        "Recall (bad)": recall_score(y_test, pred, zero_division=0),
        "Precision (bad)": precision_score(y_test, pred, zero_division=0),
        "F1 (bad)": f1_score(y_test, pred, zero_division=0),
        "Cost @ best t": cost_of(y_test, pred),
        "Cost @ 0.5": cost_of(y_test, (proba >= 0.5).astype(int)),
    })

summary = pd.DataFrame([{"model": n, "CV AUC": r["cv_auc"], "OOF AUC": r["oof_auc"],
                         "threshold": r["threshold"], **r["test_metrics"]}
                        for n, r in results.items()]).set_index("model")
best_name = summary["CV AUC"].idxmax()
print("โมเดลที่เลือก (CV AUC สูงสุด):", best_name)
display(summary.round(3))

best = results[best_name]
ConfusionMatrixDisplay(confusion_matrix(y_test, best["pred_test"]),
                       display_labels=["good", "bad"]).plot(cmap="Blues")
plt.title(f"{best_name} @ threshold {best['threshold']:.2f}"); plt.show()

## 3. Permutation importance บนชุดทดสอบ

In [ ]:
for name, r in results.items():
    pi = permutation_importance(r["pipeline"], X_test, y_test, scoring="roc_auc",
                                n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
    r["perm_importance"] = (pd.DataFrame({"feature": X_test.columns,
                                          "importance_mean": pi.importances_mean,
                                          "importance_std": pi.importances_std})
                            .sort_values("importance_mean", ascending=False)
                            .reset_index(drop=True))

imp = results[best_name]["perm_importance"]
top = imp.head(15).iloc[::-1]
plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"])
plt.xlabel("decrease in ROC-AUC"); plt.title(f"Permutation importance — {best_name}")
plt.tight_layout(); plt.show()
imp.head(10)

## 4. Fairness metrics ตาม sex และ age_group
ผลลัพธ์ที่เป็นประโยชน์ต่อผู้กู้คือ **ได้รับอนุมัติ (ทำนายว่า good)**

| ตัวชี้วัด | ความหมาย | ค่าที่เป็นธรรม |
|---|---|---|
| Demographic parity difference | ความต่างของอัตราอนุมัติระหว่างกลุ่ม | ใกล้ 0 |
| Disparate impact ratio | อัตราอนุมัติต่ำสุด / สูงสุด | ≥ 0.8 |
| Equal opportunity difference (TPR) | ความต่างของอัตราการตรวจพบผู้ที่ bad จริง | ใกล้ 0 |
| FPR difference | ความต่างของอัตราปฏิเสธผู้ที่ good จริง (เทียบเท่า equal opportunity ในมุมผลลัพธ์ "อนุมัติ") | ใกล้ 0 |
| Equalized odds difference | ค่ามากสุดของความต่าง TPR และ FPR | ใกล้ 0 |

In [ ]:
def group_metrics(y_true, y_pred, groups):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rows = []
    for g in sorted(groups.unique()):
        m = (groups == g).values
        yt, yp = y_true[m], y_pred[m]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        rows.append({"group": g, "n": int(m.sum()),
                     "actual_bad_rate": yt.mean(), "pred_bad_rate": yp.mean(),
                     "approval_rate": 1 - yp.mean(),
                     "TPR": tp / (tp + fn) if (tp + fn) else np.nan,
                     "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
                     "precision_bad": tp / (tp + fp) if (tp + fp) else np.nan})
    return pd.DataFrame(rows)

def fairness_summary(tbl):
    ar, tpr, fpr = tbl["approval_rate"], tbl["TPR"], tbl["FPR"]
    return {"Demographic parity diff.": ar.max() - ar.min(),
            "Disparate impact ratio": ar.min() / ar.max() if ar.max() > 0 else np.nan,
            "Equal opportunity diff. (TPR)": tpr.max() - tpr.min(),
            "FPR diff.": fpr.max() - fpr.min(),
            "Equalized odds diff.": max(tpr.max() - tpr.min(), fpr.max() - fpr.min())}

fair_rows = []
for attr in ["sex", "age_group"]:
    tbl = group_metrics(y_test.values, best["pred_test"], groups_test[attr])
    print(f"\n=== {attr} ==="); display(tbl.round(3))
    fair_rows.append({"attribute": attr, "features": "รวมตัวแปรอ่อนไหว", **fairness_summary(tbl)})
pd.DataFrame(fair_rows).set_index("attribute").round(3)

### 4.1 เปรียบเทียบ: ตัด personal_status และ age ออกจากฟีเจอร์

In [ ]:
drop = ["personal_status", "age"]
Xtr2, Xte2 = X_train.drop(columns=drop), X_test.drop(columns=drop)
pipe2 = clone(best["pipeline"]).set_params(prep=build_preprocessor(Xtr2))
pipe2.fit(Xtr2, y_train)
oof2 = cross_val_predict(pipe2, Xtr2, y_train, cv=skf, method="predict_proba")[:, 1]
t2, _ = find_best_threshold(y_train.values, oof2)
proba2 = pipe2.predict_proba(Xte2)[:, 1]
pred2 = (proba2 >= t2).astype(int)
print(f"ไม่ใช้ตัวแปรอ่อนไหว: test AUC = {roc_auc_score(y_test, proba2):.3f}, "
      f"threshold = {t2:.2f}, cost = {cost_of(y_test, pred2)}")

for attr in ["sex", "age_group"]:
    fs = fairness_summary(group_metrics(y_test.values, pred2, groups_test[attr]))
    fair_rows.append({"attribute": attr, "features": "ตัดตัวแปรอ่อนไหว", **fs})
fairness_compare = pd.DataFrame(fair_rows).sort_values(["attribute", "features"]).reset_index(drop=True)
fairness_compare.round(3)

## 5. บันทึกไฟล์สำหรับ deploy
- `german_credit.csv` ข้อมูลดิบสำหรับหน้า EDA
- `model_artifacts.joblib` Pipeline ทั้ง 4 โมเดล, threshold, OOF, ผล CV/ทดสอบ, importance, fairness
- `requirements.txt` ระบุเวอร์ชันให้ตรงกับ Colab (ไฟล์ joblib ต้องโหลดด้วย scikit-learn เวอร์ชันเดียวกัน)

> `app.py`, `train_model.py` และ `.streamlit/config.toml` อยู่ใน GitHub repository แล้ว ไม่ต้องสร้างจาก Notebook

In [ ]:
os.makedirs("deploy", exist_ok=True)
df.drop(columns=["sex", "age_group"]).to_csv("deploy/german_credit.csv", index=False)

artifacts = {
    "meta": {"best_model": best_name, "random_state": RANDOM_STATE, "test_size": TEST_SIZE,
             "n_splits": N_SPLITS, "cost_fn": COST_FN, "cost_fp": COST_FP,
             "versions": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__,
                          "numpy": np.__version__}},
    "results": results,
    "X_train": X_train, "y_train": y_train,
    "X_test": X_test, "y_test": y_test,
    "groups_test": groups_test, "fairness_compare": fairness_compare,
}
joblib.dump(artifacts, "deploy/model_artifacts.joblib", compress=3)

with open("deploy/requirements.txt", "w") as f:
    f.write(f"streamlit>=1.50\nscikit-learn=={sklearn.__version__}\n"
            f"pandas=={pd.__version__}\nnumpy=={np.__version__}\nplotly>=5.20\njoblib\n")

for fn in sorted(os.listdir("deploy")):
    print(f"{fn:28s} {os.path.getsize('deploy/' + fn) / 1024:,.1f} KB")

### 5.1 ทดสอบว่าไฟล์โหลดได้

In [ ]:
chk = joblib.load("deploy/model_artifacts.joblib")
print("โมเดลในไฟล์:", list(chk["results"]))
for n, r in chk["results"].items():
    p = r["pipeline"].predict_proba(chk["X_test"].head(3))[:, 1]
    print(f"{n:24s} threshold={r['threshold']:.2f}  p(bad)={np.round(p, 3)}")

### 5.2 ดาวน์โหลดไฟล์

In [ ]:
shutil.make_archive("deploy_files", "zip", "deploy")
try:
    from google.colab import files
    files.download("deploy_files.zip")
except ImportError:
    print("ไม่ได้รันใน Colab — ไฟล์อยู่ที่ deploy_files.zip")

## 6. ขั้นตอน deploy
1. แตก `deploy_files.zip` → นำ 3 ไฟล์ (`german_credit.csv`, `model_artifacts.joblib`, `requirements.txt`) ไปวางที่ root ของ repository (ทับ requirements.txt เดิม)
2. Commit ขึ้น GitHub → https://share.streamlit.io → Create app → main file = `app.py`
3. Advanced settings → Python ให้ตรงกับ Colab (ดูจากเซลล์ด้านล่าง)

In [ ]:
!python --version
!pip list | grep -iE "^(scikit-learn|numpy|pandas|joblib) "